In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
X_train = pd.read_csv("../data/X_rf_train.csv")
X_test = pd.read_csv("../data/X_rf_test.csv")
y_train = pd.read_csv("../data/y_train.csv")
y_test = pd.read_csv("../data/y_test.csv")

In [3]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [4]:
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    oob_score=True
)

regressor.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [5]:
##close to 36% of the data from training set is not used when training random forest or an bagging algorithm where bootstrapping is being done so this is the r2 score of testing that 36% of the data
print("Out-of-Bag Score:", regressor.oob_score_)

Out-of-Bag Score: 0.9654489906087458


In [6]:
y_pred = regressor.predict(X_test)

In [7]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))

In [8]:
y_train_pred = regressor.predict(X_train)
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.9951


In [9]:
print("TEST METRICS ARE")
print("R2 :", round(r2,4))
print("MAE :", round(mae,4))
print("RMSE :", round(rmse,4))

TEST METRICS ARE
R2 : 0.952
MAE : 0.0379
RMSE : 0.1258


In [10]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(regressor,X_train,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.9694272  0.95061731 0.97579551 0.96872978 0.95468578 0.95671722
 0.95164383 0.95184988 0.97045047 0.95707707]

Mean CV R2: 0.9607
Std CV R2: 0.0089


In [11]:
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score

cv = RepeatedKFold(
    n_splits=10,
    n_repeats=10,
    random_state=42
)

scores = cross_val_score(
    regressor,
    X_train,
    y_train,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print(scores.mean())
print(scores.std())

0.9627739930111635
0.0171507844829782


In [12]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, val_scores = learning_curve(
    estimator=regressor,
    X=X_train,
    y=y_train,
    cv=10,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

print("Train Sizes:")
print(train_sizes)

print()

print("Train R2:")
print(train_mean)

print()

print("Validation R2:")
print(val_mean)

Train Sizes:
[ 385  770 1155 1540 1925 2310 2695 3080 3465 3851]

Train R2:
[0.9638611  0.97736966 0.98845058 0.99058957 0.99187619 0.99329377
 0.99347037 0.99424997 0.99452418 0.99474067]

Validation R2:
[0.81429682 0.88652356 0.91689533 0.93849628 0.94499757 0.95078916
 0.95607168 0.9561564  0.95704595 0.96060189]


In [37]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf = RandomForestRegressor(
    random_state=42
)

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

print("Best Parameters:")
print(rf_search.best_params_)

print("Best CV Score:")
print(rf_search.best_score_)

Best Parameters:
{'n_estimators': 700, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}
Best CV Score:
0.9552909385205555


In [ ]:
best_rf = rf_search.best_estimator_

y_pred = best_rf.predict(X_test)

from sklearn.metrics import r2_score

print("Test R2:")
print(r2_score(y_test, y_pred))

Test R2:
0.9680964051771699


: 

In [17]:
df=pd.read_csv("../data/DataWithDescriptors.csv")
df.columns

Index(['Unnamed: 0', 'IL NAME', 'CATION NAME', 'ANION NAME', 'Cation_SMILES',
       'Anion_SMILES', 'Temperature (0 C)', 'Pressure (bar)',
       'CO2 capacity (mol CO2/mol IL)', 'Cation_Family',
       ...
       'An_fr_sulfide', 'An_fr_sulfonamd', 'An_fr_sulfone',
       'An_fr_term_acetylene', 'An_fr_tetrazole', 'An_fr_thiazole',
       'An_fr_thiocyan', 'An_fr_thiophene', 'An_fr_unbrch_alkane',
       'An_fr_urea'],
      dtype='str', length=444)

In [18]:
features=pd.read_excel("../data/RF_Top20_Features.xlsx")
features

,Descriptor
0,Pressure (bar)
1,Temperature (0 C)
2,Cat_MolLogP
3,Cat_EState_VSA2
4,An_TPSA
5,Cat_VSA_EState3
6,An_AvgIpc
7,An_SlogP_VSA1
8,Cat_TPSA
9,Cat_EState_VSA9


In [20]:
df.describe()

,Unnamed: 0,Temperature (0 C),Pressure (bar),CO2 capacity (mol CO2/mol IL),Cat_MaxAbsEStateIndex,Cat_MaxEStateIndex,Cat_MinAbsEStateIndex,Cat_MinEStateIndex,Cat_qed,Cat_SPS,...,An_fr_sulfide,An_fr_sulfonamd,An_fr_sulfone,An_fr_term_acetylene,An_fr_tetrazole,An_fr_thiazole,An_fr_thiocyan,An_fr_thiophene,An_fr_unbrch_alkane,An_fr_urea
count,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,5349.000000,...,5349.000000,5349.000000,5349.0,5349.0,5349.0,5349.0,5349.000000,5349.0,5349.000000,5349.0
mean,2674.000000,43.361554,16.809779,0.351997,3.113498,3.113498,1.028575,0.863775,0.500547,11.559203,...,0.000187,0.772107,0.0,0.0,0.0,0.0,0.004300,0.0,0.010282,0.0
std,1544.267626,21.041986,28.551239,0.556316,2.119076,2.119076,0.295891,1.074778,0.078408,3.697888,...,0.013673,0.973777,0.0,0.0,0.0,0.0,0.065438,0.0,0.113121,0.0
min,0.000000,4.970000,0.000013,0.000000,2.125000,2.125000,0.122336,-7.856082,0.076609,6.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
25%,1337.000000,25.000000,2.000000,0.070664,2.211806,2.211806,1.056389,1.056389,0.467050,10.200000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
50%,2674.000000,40.000000,8.442000,0.186000,2.255899,2.255899,1.150139,1.150139,0.495865,10.285714,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
75%,4011.000000,59.980000,17.100000,0.412429,2.414768,2.414768,1.167500,1.167500,0.554691,10.714286,...,0.000000,2.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0
max,5348.000000,139.950000,345.500000,8.293680,13.495084,13.495084,1.366806,1.366806,0.758479,30.454545,...,1.000000,2.000000,0.0,0.0,0.0,0.0,1.000000,0.0,4.000000,0.0


In [21]:
rf_features = features.iloc[:,0].tolist()

print(rf_features)
print(len(rf_features))

['Pressure (bar)', 'Temperature (0 C)', 'Cat_MolLogP', 'Cat_EState_VSA2', 'An_TPSA', 'Cat_VSA_EState3', 'An_AvgIpc', 'An_SlogP_VSA1', 'Cat_TPSA', 'Cat_EState_VSA9', 'Cat_BCUT2D_LOGPLOW', 'Cat_NumHDonors', 'Cat_SMR_VSA5', 'Cat_SMR_VSA1', 'Cat_PEOE_VSA1', 'Cat_NHOHCount', 'Cat_EState_VSA8', 'Cat_Chi4v', 'An_BalabanJ', 'An_PEOE_VSA3']
20


In [26]:
ranged_df=df[(df["Temperature (0 C)"] >= 25) &(df["Temperature (0 C)"] <= 30) &(df["Pressure (bar)"] >= 0.5) &(df["Pressure (bar)"] <= 1.5)].copy()

In [27]:
ranged_df.describe()

,Unnamed: 0,Temperature (0 C),Pressure (bar),CO2 capacity (mol CO2/mol IL),Cat_MaxAbsEStateIndex,Cat_MaxEStateIndex,Cat_MinAbsEStateIndex,Cat_MinEStateIndex,Cat_qed,Cat_SPS,...,An_fr_sulfide,An_fr_sulfonamd,An_fr_sulfone,An_fr_term_acetylene,An_fr_tetrazole,An_fr_thiazole,An_fr_thiocyan,An_fr_thiophene,An_fr_unbrch_alkane,An_fr_urea
count,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,...,146.000000,146.000000,146.0,146.0,146.0,146.0,146.0,146.0,146.000000,146.0
mean,2130.917808,26.805822,0.937501,0.233495,3.115569,3.115569,1.035509,0.733522,0.475782,11.631347,...,0.006849,0.506849,0.0,0.0,0.0,0.0,0.0,0.0,0.068493,0.0
std,1585.237254,2.370709,0.277405,0.389334,1.868038,1.868038,0.252641,1.289823,0.148699,3.756898,...,0.082761,0.872939,0.0,0.0,0.0,0.0,0.0,0.0,0.400991,0.0
min,14.000000,25.000000,0.500000,0.003412,2.125000,2.125000,0.122336,-7.856082,0.076609,9.428571,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
25%,374.250000,25.000000,0.751500,0.021946,2.211806,2.211806,1.056389,1.056389,0.461795,9.909091,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
50%,2182.500000,25.010000,0.999000,0.039506,2.365434,2.365434,1.150139,1.150139,0.541779,10.200000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
75%,3458.750000,30.000000,1.000000,0.214690,3.674173,3.674173,1.155926,1.155926,0.578815,11.583333,...,0.000000,1.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
max,4971.000000,30.000000,1.500000,2.050000,13.495084,13.495084,1.364796,1.364796,0.758479,24.666667,...,1.000000,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,4.000000,0.0


In [28]:
ranged_df["RF_Pred"] = regressor.predict(ranged_df[rf_features])

In [29]:
print(
    "Unique ILs:",
    ranged_df["IL NAME"].nunique()
)

Unique ILs: 72


In [35]:
rf_ranking = (ranged_df.groupby("IL NAME")["RF_Pred"].mean().reset_index())

In [36]:
top5_rf = rf_ranking.sort_values(by="RF_Pred",ascending=False).head(10)

print(top5_rf)

          IL NAME   RF_Pred
68    [TETAH][Pz]  1.847804
67    [TETAH][Py]  1.622230
69    [TETAH][Tz]  1.529310
66    [TETAH][Im]  1.505549
57  [P66614][Gly]  1.199050
1     [AmemI][Br]  0.942643
58  [P66614][Ile]  0.923486
62  [P66614][Sar]  0.897215
71  [aP4443][Ala]  0.890461
7     [BMIM][Gly]  0.875464


In [13]:
X_train = pd.read_csv("../data/X_rf_trainpKa.csv")
X_test = pd.read_csv("../data/X_rf_testpKa.csv")
y_train = pd.read_csv("../data/y_trainpKa.csv")
y_test = pd.read_csv("../data/y_testpKa.csv")

In [14]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [15]:
from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    oob_score=True
)

regressor.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [16]:
##close to 36% of the data from training set is not used when training random forest or an bagging algorithm where bootstrapping is being done so this is the r2 score of testing that 36% of the data
print("Out-of-Bag Score:", regressor.oob_score_)

Out-of-Bag Score: 0.96383496508583


In [17]:
y_pred = regressor.predict(X_test)

In [18]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))

In [19]:
y_train_pred = regressor.predict(X_train)
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.9949


In [20]:
print("TEST METRICS ARE")
print("R2 :", round(r2,4))
print("MAE :", round(mae,4))
print("RMSE :", round(rmse,4))


TEST METRICS ARE
R2 : 0.9525
MAE : 0.038
RMSE : 0.1251


In [21]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(regressor,X_train,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.97275513 0.95640883 0.97451074 0.97225338 0.95368794 0.94907712
 0.95903918 0.95137343 0.97056538 0.95350994]

Mean CV R2: 0.9613
Std CV R2: 0.0095
